In [1]:
# ==========================================
# 1. PREPARACIÓ DE L'ENTORN MODERN
# ==========================================
using Pkg
Pkg.activate("/home/antonibancells/Desktop/Codi/TFM/ProvesModernes")

  Activating project at `~/Desktop/Codi/TFM/ProvesModernes`


In [2]:
# Librerías necesarias
using ITensors
using ITensorMPS
using Plots
using LinearAlgebra

En este trabajo se estudia el problema de contorno dado por:
		$$\frac{1}{\alpha} \frac{\partial T(x,t)}{\partial t}=\frac{\partial^2 T(x,t)}{\partial x^2}-\sigma^2 T(x,t)+f(x) \ , \ 0<x<L \ , \ 0<t<t_{max}$$
		$$\frac{\partial T(0,t)}{\partial x}=\beta T(0)-\gamma $$
		$$T(L,t)=\delta \ , \ 0<t<t_{max} $$
		$$f(x)=F_0 e^{-\mu x} \ , \ 0<t<t_{max}$$
La parte espacial se discretiza mediante diferencias finitas de segundo orden. La condició de Robin se discretiza con la ayuda del nodo fantasma. La integración temporal se realiza mediante el métod de Crank-Nicolson, pero los vectores de temperatura se representan mediante MPS, las matrices son operadores en formato MPO y los sistemas de ecuaciones se resuelven mediante DMRG.

In [3]:
# =============================================================================
# 1. PARÁMETROS DEL PROBLEMA Y CONFIGURACIÓN
# =============================================================================
W = 1.0         
σ = 2.0         
β = 1.5         
γ = 0.5         
δ = 3.0         
F0 = 10.0       
μ = 1.0         
alpha = 2

dt = 0.0025
t_final = 0.25
pasos_t = Int(t_final / dt)

L_sites = 8
N = 2^L_sites   
dx = W / N

println("--- Evolución Temporal QTT-DMRG (Crank-Nicolson) ---")
println("Qubits: $L_sites | Nodes: $N | Pasos de tiempo: $pasos_t")
println("---------------------------------------------------")

sites = siteinds("Qubit", L_sites)

--- Evolución Temporal QTT-DMRG (Crank-Nicolson) ---
Qubits: 8 | Nodes: 256 | Pasos de tiempo: 100
---------------------------------------------------


8-element Vector{Index{Int64}}:
 (dim=2|id=960|"Qubit,Site,n=1")
 (dim=2|id=641|"Qubit,Site,n=2")
 (dim=2|id=667|"Qubit,Site,n=3")
 (dim=2|id=898|"Qubit,Site,n=4")
 (dim=2|id=687|"Qubit,Site,n=5")
 (dim=2|id=217|"Qubit,Site,n=6")
 (dim=2|id=487|"Qubit,Site,n=7")
 (dim=2|id=330|"Qubit,Site,n=8")

**Construcción Modular del MPS de Fuentes (Opción 1)**

Esta aproximación divide el vector de fuentes físicas en dos componentes independientes representadas como Matrix Product States (MPS), los cuales se combinan posteriormente.

**Fondo Exponencial Puro (mps_exponencial_lsb)**
Genera el perfil de atenuación de la fuente externa $f(x) = A_{\text{amp}} e^{-\mu_{\text{exp}} x}$.

*Fundamento matemático:* En ordenación de bit menos significativo (LSB), cualquier coordenada de la malla discreta se descompone según sus bits físicos $b_i \in \{0, 1\}$ como $x = dx \sum_{i=1}^L b_i 2^{i-1}$. Al sustituir esto en la exponencial, la función se factoriza analíticamente en un producto directo de tensores locales independientes:$$f(x) = A_{\text{amp}} \prod_{i=1}^L e^{-\mu_{\text{exp}} \cdot dx \cdot b_i 2^{i-1}}$$

*Estructura del MPS:* Al tratarse de un estado de producto puro (sin entrelazamiento virtual), la dimensión de enlace interna es exactamente $D = 1$. Cada sitio $i$ evalúa el bit físico de forma local: si el qubit es $|0\rangle$ inyecta $1.0$, y si es $|1\rangle$ inyecta $e^{-\mu_{\text{exp}} \cdot dx \cdot 2^{i-1}}$. La amplitud global $A_{\text{amp}}$ se introduce únicamente en el primer sitio para evitar acumulaciones multiplicativas.

**Corrección Automática de Fronteras (mps_correccion_fronteras_lsb)**
Aísla exclusivamente los extremos absolutos de la malla (el primer elemento $|00\dots0\rangle$ y el último $|11\dots1\rangle$) para aplicar las correcciones de contorno $\Delta L$ y $\Delta R$.

*Mecanismo del autómata:* Utiliza una dimensión de enlace virtual fija $D = 2$ para abrir dos canales de control de memoria que actúan como "filtros":
- Canal 1 (Filtro de Ceros): Se propaga únicamente a través del bit físico $|0\rangle$. Si detecta un solo bit $1$, el canal se destruye. Al llegar al sitio final $L$, inyecta la corrección izquierda $\Delta L$.
- Canal 2 (Filtro de Unos): Se propaga únicamente a través del bit físico $|1\rangle$. Si detecta un solo bit $0$, el canal se destruye. Al llegar al sitio final $L$, inyecta la corrección derecha $\Delta R$.
**Unificación por Compresión Variacional**
Tras generar ambos estados por separado, el script ejecuta la suma de los MPS mediante la instrucción de ITensor en Julia:

mps_fuentes = add(mps_f, mps_bc; cutoff=1e-15)

Esta operación fusiona los espacios virtuales en una red intermedia con dimensión de enlace acumulada $D = 1 + 2 = 3$. Acto seguido, ITensor realiza de forma automática un barrido de ortogonalización y compresión mediante SVD para eliminar las redundancias numéricas bajo el umbral (cutoff) definido.

In [4]:
# =============================================================================
# 2. DEFINICIÓN DEL MPS DE LAS FUENTES, POR SEPARADO FONDO EXPONENCIAL Y 
#    CONDICIONES DE CONTORNO. MÁS ADELANTE SE SUMAN
# =============================================================================
function mps_exponencial_lsb(sites, A_amp, μ_exp, dx)
    L = length(sites)
    mps = MPS(sites)
    links = [Index(1, "Link,l=$i") for i in 1:(L-1)]
    
    for i in 1:L
        val_0, val_1 = 1.0, exp(-μ_exp * dx * 2^(i-1))
        if i == 1
            A = ITensor(sites[1], links[1])
            A[sites[1]=>1, links[1]=>1] = A_amp * val_0
            A[sites[1]=>2, links[1]=>1] = A_amp * val_1
            mps[1] = A
        elseif i == L
            A = ITensor(links[L-1], sites[L])
            A[links[L-1]=>1, sites[L]=>1] = val_0
            A[links[L-1]=>1, sites[L]=>2] = val_1
            mps[L] = A
        else
            A = ITensor(links[i-1], sites[i], links[i])
            A[links[i-1]=>1, sites[i]=>1, links[i]=>1] = val_0
            A[links[i-1]=>1, sites[i]=>2, links[i]=>1] = val_1
            mps[i] = A
        end
    end
    return mps
end

function mps_correccion_fronteras_lsb(sites, ΔL, ΔR)
    L = length(sites)
    mps = MPS(sites)
    links = [Index(2, "Link,l=$i") for i in 1:(L-1)]
    
    A1 = ITensor(sites[1], links[1])
    A1[sites[1]=>1, links[1]=>1] = 1.0; A1[sites[1]=>2, links[1]=>2] = 1.0
    mps[1] = A1
    
    for i in 2:(L-1)
        Ai = ITensor(links[i-1], sites[i], links[i])
        Ai[links[i-1]=>1, sites[i]=>1, links[i]=>1] = 1.0
        Ai[links[i-1]=>2, sites[i]=>2, links[i]=>2] = 1.0
        mps[i] = Ai
    end
    
    AL = ITensor(links[L-1], sites[L])
    AL[links[L-1]=>1, sites[L]=>1] = ΔL
    AL[links[L-1]=>2, sites[L]=>2] = ΔR
    mps[L] = AL
    return mps
end


mps_correccion_fronteras_lsb (generic function with 1 method)

**Definición del MPO Tridiagonal (Ordenación LSB)**

Esta función genera de forma analítica el Matrix Product Operator (MPO) tridiagonal del sistema con una dimensión de enlace virtual exacta $D = 4$. Al operar en formato de bit menos significativo (LSB), los enlaces virtuales actúan como un autómata que gestiona la aritmética binaria del movimiento entre nodos contiguos de la malla.

**Los 4 Canales Virtuales**

La lógica de los tensores se organiza bajo el siguiente significado para cada índice del enlace virtual (link):
- Canal 1 (Acumulador / Resuelto): Recoge los elementos calculados de la diagonal principal (parámetro $-a$) y los términos de interacción lateral (hoppings) que ya se han cerrado. Se propaga hacia el final de la cadena mediante la Identidad ($I$).
- Canal 2 (Acarreo de Subida): Inicia una transición aritmética en el primer qubit mediante $S^+$. Se propaga por el interior a través de operadores $S^+$ y se resuelve sumándose al Canal 1 bajo el operador $b \cdot S^-$.
- Canal 3 (Acarreo de Bajada): Inicia la transición complementaria mediante $S^-$. Se propaga en el bulk mediante operadores $S^-$ y se resuelve hacia el Canal 1 bajo el operador $b \cdot S^+$.
- Canal 4 (Filtro de Robin): Monitorea exclusivamente el camino de puros ceros ($|00\dots0\rangle$) aplicando el proyector $P_0$. En el último sitio aplica de forma exacta la penalización de frontera $-c$.

**Representación Matricial de los Tensores**

La estructura algebraica del autómata programado se puede resumir visualmente en las siguientes matrices de operadores:
1. Sitio Inicial (Vector Fila $1 \times 4$)Inyecta las interacciones locales del primer bit y abre los canales de acarreo y el filtro de frontera:
$$W_1 = \begin{pmatrix} -aI + bS^+ + bS^- & S^+ & S^- & P_0 \end{pmatrix}$$
2. Sitios del Bulk (Matrices $4 \times 4$)Controlan la propagación de las transiciones binarias a lo largo de la cadena. Las filas representan el canal de entrada y las columnas el canal de salida:$$W_i = \begin{pmatrix} 
I & 0 & 0 & 0 \\ 
bS^- & S^+ & 0 & 0 \\ 
bS^+ & 0 & S^- & 0 \\ 
0 & 0 & 0 & P_0 
\end{pmatrix}$$
3. Sitio Final o Cierre (Vector Columna $4 \times 1$)Recibe y sella todas las operaciones del autómata, aplicando la condición de Robin en la rama del filtro:$$W_L = \begin{pmatrix} I \\ bS^- \\ bS^+ \\ -cP_0 \end{pmatrix}$$

In [5]:
# =============================================================================
# 3. DEFINICIÓN DEL MPO TRIDIAGONAL
# =============================================================================

function construir_mpo_manual_perfecto_lsb(sites, a::Float64, b::Float64, c::Float64)
    L = length(sites)
    MPO_manual = MPO(sites)
    links = [Index(4, "Link,l=$l") for l in 1:(L-1)]
    Id = [1.0 0.0; 0.0 1.0]; P0 = [1.0 0.0; 0.0 0.0]; 
    Sp = [0.0 1.0; 0.0 0.0]; Sm = [0.0 0.0; 1.0 0.0]
    
    W1 = ITensor(sites[1], prime(sites[1]), links[1])
    for s in 1:2, sp in 1:2
        W1[sites[1]=>s, prime(sites[1])=>sp, links[1]=>1] = -a * Id[s, sp] + b * Sp[s, sp] + b * Sm[s, sp]
        W1[sites[1]=>s, prime(sites[1])=>sp, links[1]=>2] = Sp[s, sp]
        W1[sites[1]=>s, prime(sites[1])=>sp, links[1]=>3] = Sm[s, sp]
        W1[sites[1]=>s, prime(sites[1])=>sp, links[1]=>4] = P0[s, sp]
    end
    MPO_manual[1] = W1
    
    for i in 2:(L-1)
        W = ITensor(links[i-1], sites[i], prime(sites[i]), links[i])
        for s in 1:2, sp in 1:2
            W[links[i-1]=>1, sites[i]=>s, prime(sites[i])=>sp, links[i]=>1] = Id[s, sp]
            W[links[i-1]=>2, sites[i]=>s, prime(sites[i])=>sp, links[i]=>1] = b * Sm[s, sp]
            W[links[i-1]=>2, sites[i]=>s, prime(sites[i])=>sp, links[i]=>2] = Sp[s, sp]
            W[links[i-1]=>3, sites[i]=>s, prime(sites[i])=>sp, links[i]=>1] = b * Sp[s, sp]
            W[links[i-1]=>3, sites[i]=>s, prime(sites[i])=>sp, links[i]=>3] = Sm[s, sp]
            W[links[i-1]=>4, sites[i]=>s, prime(sites[i])=>sp, links[i]=>4] = P0[s, sp]
        end
        MPO_manual[i] = W
    end
    
    WN = ITensor(links[L-1], sites[L], prime(sites[L]))
    for s in 1:2, sp in 1:2
        WN[links[L-1]=>1, sites[L]=>s, prime(sites[L])=>sp] = Id[s, sp]
        WN[links[L-1]=>2, sites[L]=>s, prime(sites[L])=>sp] = b * Sm[s, sp]
        WN[links[L-1]=>3, sites[L]=>s, prime(sites[L])=>sp] = b * Sp[s, sp]
        WN[links[L-1]=>4, sites[L]=>s, prime(sites[L])=>sp] = -c * P0[s, sp]
    end
    MPO_manual[L] = WN
    return MPO_manual
end

construir_mpo_manual_perfecto_lsb (generic function with 1 method)

In [6]:
# =============================================================================
# 4. CONSTRUCCIÓN DEL TÉRMINO FUENTE b EN FORMATO MPS
# =============================================================================

# Construcció del terme font b (MPS)
ΔL = - (F0 / 2) - (γ / dx) + F0
val_f_N = -F0 * exp(-μ * (N-1) * dx)
ΔR = (val_f_N - δ / dx^2) - val_f_N

mps_f = mps_exponencial_lsb(sites, -F0, μ, dx)
mps_bc = mps_correccion_fronteras_lsb(sites, ΔL, ΔR)
mps_objetivo = add(mps_f, mps_bc; cutoff=1e-15)

MPS
[1] ((dim=2|id=960|"Qubit,Site,n=1"), (dim=3|id=96|"Link,l=1"))
[2] ((dim=2|id=641|"Qubit,Site,n=2"), (dim=3|id=162|"Link,l=2"), (dim=3|id=96|"Link,l=1"))
[3] ((dim=2|id=667|"Qubit,Site,n=3"), (dim=3|id=953|"Link,l=3"), (dim=3|id=162|"Link,l=2"))
[4] ((dim=2|id=898|"Qubit,Site,n=4"), (dim=3|id=762|"Link,l=4"), (dim=3|id=953|"Link,l=3"))
[5] ((dim=2|id=687|"Qubit,Site,n=5"), (dim=3|id=706|"Link,l=5"), (dim=3|id=762|"Link,l=4"))
[6] ((dim=2|id=217|"Qubit,Site,n=6"), (dim=3|id=337|"Link,l=6"), (dim=3|id=706|"Link,l=5"))
[7] ((dim=2|id=487|"Qubit,Site,n=7"), (dim=2|id=206|"Link,l=7"), (dim=3|id=337|"Link,l=6"))
[8] ((dim=2|id=330|"Qubit,Site,n=8"), (dim=2|id=206|"Link,l=7"))


**MPOs Temporales Reescalados (Esquema de Crank-Nicolson)**

Este bloque de código realiza el reescalado algebraico de los coeficientes de la matriz espacial $M$ para inyectar la identidad y los pasos temporales directamente en las amplitudes de los MPOs. De este modo, se construyen analíticamente los operadores algebraicos del paso implícito y explícito, evitando realizar sumas directas de tensores (operación computacionalmente costosa que incrementaría de manera artificial la dimensión de enlace virtual $D$).El esquema temporal de Crank-Nicolson para un paso $t_n \to t_{n+1}$ viene dado por:
$$\left( I - \frac{\alpha \Delta t}{2} M \right) T^{n+1} = \left( I + \frac{\alpha \Delta t}{2} M \right) T^n + \alpha \Delta t \cdot S^{n+1/2}$$
Definiendo el factor de ponderación común como $C = \frac{\alpha \Delta t}{2}$ (coeff), el algoritmo sintoniza los parámetros físicos para reutilizar la función constructora del MPO tridiagonal manual en dos configuraciones independientes:
1. Operador Implícito o del Lado Izquierdo ($M_{\text{lhs}} = I - C \cdot M$)
Este MPO actúa sobre el vector de temperaturas futuro $T^{n+1}$. Como la función construir_mpo_manual_perfecto_lsb asume por definición los signos como $-a$ para la diagonal principal y $-c$ para la corrección de Robin, los parámetros del miembro izquierdo se reescalan invirtiendo el signo del factor temporal $C$:
    - Diagonal Principal (a_lhs): Alberga la contribución de la identidad y el decaimiento de la matriz: $a_{\text{lhs}} = -(1 - C \cdot \text{val}_a)$.
    - Super/Subdiagonal (b_lhs): Controla los acoplamientos y la difusión espacial adyacente: $b_{\text{lhs}} = -C \cdot \text{val}_b$.
    - Frontera de Robin (c_lhs): Modifica localmente el elemento $(1,1)$ para ajustar la condición de contorno implícita: $c_{\text{lhs}} = -C \cdot \text{val}_c$

2. Operador Explícito o del Lado Derecho ($M_{\text{rhs}} = I + C \cdot M$)
Este MPO se aplica como un operador de contracción directa sobre el estado térmico actual $T^n$. Conserva el signo positivo de la formulación original:
    - Diagonal Principal (a_rhs): $a_{\text{rhs}} = -(1 + C \cdot \text{val}_a)$.
    - Super/Subdiagonal (b_rhs): $b_{\text{rhs}} = C \cdot \text{val}_b$.
    - Frontera de Robin (c_rhs): $c_{\text{rhs}} = C \cdot \text{val}_c$.

3. Reescalado del Vector de Fuentes Temporal (mps_font_pas)
El término de la fuente externa $S$ entra en el miembro derecho de la ecuación multiplicado por el escalar global $\alpha \Delta t$. En Julia

mps_font_pas = deepcopy(mps_objetivo)
mps_font_pas[1] .*= -(alpha*dt)

**Principio de Homogeneidad en MPS:** Para multiplicar un MPS por un escalar en una red de tensores, no se altera toda la cadena. Gracias a la estructura de producto matricial, basta con multiplicar un único tensor local (en este caso el del sitio 1, mps_font_pas[1]) por dicho factor. Al contraer los índices virtuales de la red, el escalar se propaga linealmente a todo el espacio de Hilbert.
**Ajuste de Signo:** Se introduce el signo negativo porque comúnmente en los resolvedores algebraicos de tipo Krylov el miembro derecho absorbe la dirección del flujo de la fuente o se adapta a la igualdad formal del sistema lineal $A_{\text{lhs}} T^{n+1} = B_{\text{rhs}} T^n + \hat{S}$.

In [7]:
# =============================================================================
# 5. MPOs TEMPORALES REESCALADOS, CORRESPONDIENTES A LOS SISTEMAS DE Crank-Nicolson
# =============================================================================
val_a = -(2.0 / dx^2 + σ^2)
val_b = 1.0 / dx^2
A_11_real = -(1.0/dx^2 + β/dx + σ^2/2)
val_c = val_a - A_11_real

coeff = alpha*dt / 2

# M_lhs = I - coeff * A
a_lhs = -(1.0 - coeff * val_a)
b_lhs = -coeff * val_b
c_lhs = -coeff * val_c
MPO_lhs = construir_mpo_manual_perfecto_lsb(sites, a_lhs, b_lhs, c_lhs)

# M_rhs = I + coeff * A
a_rhs = -(1.0 + coeff * val_a)
b_rhs = coeff * val_b
c_rhs = coeff * val_c
MPO_rhs = construir_mpo_manual_perfecto_lsb(sites, a_rhs, b_rhs, c_rhs)

# Terme font reescalat per al pas temporal: - (alpha*dt) * b
mps_font_pas = deepcopy(mps_objetivo)
mps_font_pas[1] .*= -(alpha*dt)

ITensor ord=2 (dim=2|id=960|"Qubit,Site,n=1") (dim=3|id=96|"Link,l=1")
NDTensors.Dense{Float64, Vector{Float64}}

**Inicialización del Estado y Configuración del Modelo Clásico**

En este bloque se establece el perfil térmico inicial del sistema y se preparan las estructuras matriciales completas para resolver la evolución clásica de forma paralela, sirviendo como validación exacta frente al algoritmo cuántico (QTT).

**Inicialización del Perfil Térmico (T_mps_actual)**

Se define una condición inicial lineal entre las fronteras: $T_{\text{inicial}}(x) = x \cdot \frac{\delta}{W}$.Aunque se podría hacer una construcción manual con $\chi=2$, en este caso se inicializa un tensor hiperdenso de rango $L$ (T_tensor) asignando a cada índice espacial su equivalente binario en ordenación de bit menos significativo (LSB) mediante la función digits.La instrucción MPS(T_tensor, sites) comprime algebraicamente el estado en una cadena de tensores locales de producto matricial mediante descomposiciones SVD sucesivas.

⚠️ Nota: Esta pasarela de conversión es eficiente para mallas moderadas ($L \le 12$). Para mallas hiperdensas globales ($L > 20$), el almacenamiento del vector intermedio saturaría la memoria RAM, haciendo necesaria la inicialización manual en formato MPS.

**Matriz Clásica de Validación (A_cl)**

Se ensambla la matriz de acoplamiento espacial $M$ en un formato compacto y simétrico SymTridiagonal:
- Nodos del Bulk ($2 \le i \le N-1$): $\text{dv}_i = -\left(\frac{2}{\Delta x^2} + \sigma^2\right)$ y sub/superdiagonal $\text{ev}_i = \frac{1}{\Delta x^2}$.
- Nodo Frontera Robin ($i=1$): Incorpora la penalización del nodo fantasma:$$\text{dv}_1 = -\left(\frac{1}{\Delta x^2} + \frac{\beta}{\Delta x} + \frac{\sigma^2}{2}\right)$$

**Operadores de Crank-Nicolson y Factorización de Cholesky**
Definiendo $C = \frac{\alpha \Delta t}{2}$ (coeff), se configuran las matrices clásica implícita ($M_{\text{lhs, cl}}$) y explícita ($M_{\text{rhs, cl}}$):
$$M_{\text{lhs, cl}} = I - C \cdot A_{\text{cl}} \quad , \quad M_{\text{rhs, cl}} = I + C \cdot A_{\text{cl}}$$
Dado que $M_{\text{lhs, cl}}$ es simétrica y definida positiva, se ejecuta previamente una factorización de Cholesky:JuliaM_fact = cholesky(M_lhs_cl)
Al descomponer la matriz como $L L^T$ una única vez antes de iniciar el bucle temporal, la resolución del sistema implícito en cada iteración se reduce a dos sustituciones directas (atrás/adelante), optimizando drásticamente el tiempo de CPU del modelo de control.

In [8]:
# =============================================================================
# 6. CONDICIÓN INICIAL MPS Y EVOLUCIÓN CLáSICA EN PARALELO
# =============================================================================
# Vector clásico para inicializar (válido y eficiente para L <= 12)
T_vec_inicial = [ (i-1)*dx * (δ/W) for i in 1:N ]

# Convertir el vector clàssic a MPS en sentido LSB
T_tensor = ITensor(Float64, sites...)
for idx in 1:N
    bits = digits(idx - 1, base=2, pad=L_sites)
    T_tensor[Tuple(bits .+ 1)...] = T_vec_inicial[idx]
end
T_mps_actual = MPS(T_tensor, sites)

# Inicializar modelo clásico para la validación posterior
dv = zeros(N); ev = zeros(N - 1); b_vec = zeros(N); f_clas(x) = F0 * exp(-μ * x)
dv[1] = -(1/dx^2 + β/dx + σ^2/2); ev[1] = 1/dx^2; b_vec[1] = -f_clas(0)/2 - γ/dx
for i in 2:N-1
    dv[i] = -(2/dx^2 + σ^2); ev[i] = 1/dx^2; b_vec[i] = -f_clas((i-1)*dx)
end
dv[N] = -(2/dx^2 + σ^2); b_vec[N] = -f_clas((N-1)*dx) - δ/dx^2

A_cl = SymTridiagonal(dv, ev)
M_lhs_cl = I - coeff * A_cl; M_rhs_cl = I + coeff * A_cl
M_fact = cholesky(M_lhs_cl)
T_cl_actual = copy(T_vec_inicial)

256-element Vector{Float64}:
 0.0
 0.01171875
 0.0234375
 0.03515625
 0.046875
 0.05859375
 0.0703125
 0.08203125
 0.09375
 0.10546875
 0.1171875
 0.12890625
 0.140625
 ⋮
 2.859375
 2.87109375
 2.8828125
 2.89453125
 2.90625
 2.91796875
 2.9296875
 2.94140625
 2.953125
 2.96484375
 2.9765625
 2.98828125

**Bucle de Evolución Temporal (QTT-DMRG vs Clásico)**

Este bloque ejecuta el avance temporal $t = 0 \to t_{\max}$ en pasos_t iteraciones, resolviendo el esquema de Crank-Nicolson de forma paralela en formato cuántico (QTT) y clásico.

**Flujo del Algoritmo en QTT (Redes de Tensores)**
Para cada paso $t_m \to t_{m+1}$, el estado térmico (T_mps_actual) evoluciona en tres etapas:
- Fracción Explícita (apply): Se calcula el estado difusivo intermedio aplicando el MPO derecho sobre el MPS térmico actual con compresión SVD integrada (cutoff = 1e-12):$$\psi_{\text{parcial}} = M_{\text{rhs}} \, T^m$$
- Inyección de Fuentes (add): Se añade el vector de excitación y contornos pre-escalado mediante una suma variacional de MPSs:$$c^m = \psi_{\text{parcial}} + \hat{S}$$
- Inversión Implícita (linsolve): Se resuelve el sistema lineal variacional $M_{\text{lhs}} T^{m+1} = c^m$ mediante el solver DMRG (linsolve).
    - Ansatz: Se utiliza el propio T_mps_actual ($T^m$) como semilla de Krylov, garantizando convergencia casi instantánea por la proximidad física entre pasos temporales.
    - Hiperparámetros: Se acota la dimensión de enlace a maxdim = 40 y se realizan solo nsweeps = 3 para maximizar la velocidad.

**Modelo de Validación en Espacio Completo (Clásico)**

En paralelo, se calcula la solución clásica exacta sobre el vector denso para monitorizar el error:
- Miembro Derecho: Se evalúa la acción matricial y el vector de carga:
$$c_{\text{cl}} = M_{\text{rhs, cl}} \, T^m_{\text{cl}} + \alpha \Delta t \cdot S_{\text{cl}}$$
- Sustitución de Cholesky (\): Se invierte el sistema implícito mediante la factorización precalculada M_fact. Al reducirse a dos sustituciones directas, el cálculo es instantáneo y libre de aproximaciones variacionales, actuando como la referencia exacta.

In [9]:
# =============================================================================
# 7. BUCLE TEMPORAL
# =============================================================================
println("Iniciando bucle Crank-Nicolson temporal...")

for m in 1:pasos_t
    # 1. Aplicar parte explícita: M_rhs * T_m
    rhs_parcial = apply(MPO_rhs, T_mps_actual; cutoff=1e-12)
    
    # 2. Sumar el término fuente independiente: c_m = rhs_parcial + mps_font_pas
    c_mps = add(rhs_parcial, mps_font_pas; cutoff=1e-12)
    
    # 3. Resolver el sistema implícito: M_lhs * T_{m+1} = c_m
    # Usar el T_mps_actual existente es clave porque es una buena aproximación inicial
    T_mps_actual = linsolve(MPO_lhs, c_mps, T_mps_actual; cutoff=1e-12, maxdim=40, nsweeps=3)
    
    # Evolución clásica
    c_cl = M_rhs_cl * T_cl_actual - (alpha*dt) * b_vec
    global T_cl_actual = M_fact \ c_cl
    
    if m % 25 == 0 || m == pasos_t
        println(" - Paso $m completo (t = $(round(m*dt, digits=4)))")
    end
end

Iniciando bucle Crank-Nicolson temporal...
 - Paso 25 completo (t = 0.0625)
 - Paso 50 completo (t = 0.125)
 - Paso 75 completo (t = 0.1875)
 - Paso 100 completo (t = 0.25)


**Decodificación del MPS y Comparativa Final**

Una vez completada la evolución temporal, este bloque decodifica el estado en formato de red de tensores para reconstruir el vector físico de temperaturas y evaluar la precisión del algoritmo.

**Reconstrucción del Vector (T_resultat_dmrg)**

El perfil térmico final reside comprimido en el MPS (T_mps_actual). Para recuperar los $N = 2^L$ valores espaciales reales, el código realiza una proyección secuencial sobre la base computacional:
- Conversión LSB: Para cada nodo idx, se calcula su cadena de bits en ordenación de bit menos significativo mediante digits.
- Contracción de la Red: Se proyecta y multiplica el MPS sitio a sitio con los estados cuánticos correspondientes (bits[s] + 1). Al consumirse todos los índices virtuales, la red se reduce a un número escalar que representa la amplitud en ese punto:
$$T_{\text{DMRG}}(x_i) = \langle b_1 b_2 \dots b_L | T_{\text{mps}}\rangle$$

⚠️ Nota: Muestrear el espacio completo es útil como diagnóstico en mallas moderadas ($L \le 12$). Para mallas hiperdensas ($L > 20$), el vector denso es inmanejable en memoria, por lo que se recurre a medir observables de forma nativa sobre el MPS sin llegar a reconstruirlo.

**Validación y Error Global** 
El script muestra las primeras 5 componentes de la solución y calcula la Norma Euclídea Absoluta del residuo global en todo el dominio:
$$\text{Error Global} = \| T_{\text{clásico}} - T_{\text{DMRG}} \|_2 = \sqrt{\sum_{i=1}^N \left( T_{\text{clásico}}(x_i) - T_{\text{DMRG}}(x_i) \right)^2}$$

Residuos en el orden de $\sim 10^{-5}$ tras completar toda la evolución temporal confirman la excelente estabilidad y precisión del método. En el contexto de las redes de tensores aplicadas a ecuaciones diferenciales dinámicas, mantener un error de esta magnitud tras decenas o cientos de pasos de tiempo es un resultado óptimo. Indica que los criterios de truncamiento elegidos (cutoff = 1e-12 y maxdim = 40) logran un equilibrio perfecto: comprimen drásticamente el espacio de estados manteniendo intacta la física del problema de Bioheat sin que el error se amplifique o desestabilice el esquema de Crank-Nicolson.

In [10]:
# =============================================================================
# 8. EXTRACCIÓN Y COMPARATIVA FINAL
# =============================================================================
T_resultat_dmrg = zeros(N)
for idx in 1:N
    bits = digits(idx - 1, base=2, pad=L_sites)
    psi = T_mps_actual[1] * state(sites[1], bits[1] + 1)
    for s in 2:L_sites
        psi = psi * T_mps_actual[s] * state(sites[s], bits[s] + 1)
    end
    T_resultat_dmrg[idx] = scalar(psi)
end

println("\n==================================================")
println("COMPROVACIÓN EN t = $t_final")
println("Método Clásico: ", round.(T_cl_actual[1:5], digits=5))
println("Método QTT    : ", round.(T_resultat_dmrg[1:5], digits=5))
println("Error Euclideo: ", norm(T_cl_actual - T_resultat_dmrg))
println("==================================================")


COMPROVACIÓN EN t = 0.25
Método Clásico: [1.40453, 1.41406, 1.42025, 1.42574, 1.43134]
Método QTT    : [1.40454, 1.41405, 1.42027, 1.42574, 1.43135]
Error Euclideo: 0.00015718085980959219
